# exp04 - Perakitan tabel naskah dari berkas hasil

Notebook ini **tidak menjalankan model apa pun**. Ia hanya membaca `results/*.csv`
yang dihasilkan exp01-exp03 dan merakitnya menjadi tabel siap-tempel (Markdown dan
LaTeX). Pemisahan ini disengaja: angka pada naskah tidak boleh pernah diketik ulang
dengan tangan - itulah asal ketidakkonsistenan RMSE/MSE yang ditemukan reviewer.

Jalankan exp01, exp02, dan exp03 terlebih dahulu.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import pandas as pd
from pathlib import Path

from src.experiments import protocol as P

RESULTS = Path("../results")
OUT = RESULTS / "paper_tables"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 50)

def load(name):
    path = RESULTS / f"{name}.csv"
    if not path.exists():
        print(f"[lewati] {path} belum ada - jalankan notebook eksperimennya dulu")
        return None
    return pd.read_csv(path)

daily   = load("exp01_pharma_daily")
weekly  = load("exp02_pharma_weekly")
rossman = load("exp03_rossmann_leakage_ablation")

In [ ]:
def emit(df, stem, caption, float_fmt="%.4f"):
    """Tulis satu tabel sebagai CSV + Markdown + LaTeX dengan nama yang sama."""
    df.to_csv(OUT / f"{stem}.csv")
    (OUT / f"{stem}.md").write_text(df.to_markdown(floatfmt=".4f"), encoding="utf-8")
    (OUT / f"{stem}.tex").write_text(
        df.to_latex(float_format=float_fmt, caption=caption, label=f"tab:{stem}",
                    escape=True), encoding="utf-8")
    print(f"ditulis: {stem}.csv / .md / .tex   -- {caption}")
    return df

## Tabel 1 - PharmaSales harian: semua model di bawah protokol tunggal

In [ ]:
MODEL_ORDER = ["Naive", "SeasonalNaive(s=7)", "SeasonalNaive(s=52)", "ARIMA(5,1,0)",
               "LR", "GRNN", "P_NN", "RBFNN", "XGBoost", "LR+XGB (average)",
               "LR-XGB (residual)"]

def main_table(df, metric="test_RMSE"):
    pivot = df.pivot_table(index=["category", "feature_set"], columns="model",
                           values=metric)
    return pivot[[m for m in MODEL_ORDER if m in pivot.columns]]

if daily is not None:
    t1 = emit(main_table(daily).round(4), "tab1_pharma_daily_rmse",
              "RMSE test PharmaSales harian di bawah protokol tunggal "
              "(split 70/15/15, tuning hanya pada validation, seed 42).")
    display(t1)

## Tabel 2 - PharmaSales mingguan

In [ ]:
if weekly is not None:
    t2 = emit(main_table(weekly).round(4), "tab2_pharma_weekly_rmse",
              "RMSE test PharmaSales mingguan di bawah protokol tunggal.")
    display(t2)

## Tabel 3 - Rossmann: ablasi kebocoran `Customers`

Skala log dan skala asli dipisahkan ke dua blok kolom dengan penanda eksplisit.

In [ ]:
if rossman is not None:
    t3 = (rossman.set_index(["feature_set", "model"])
          [["n_features", "test_RMSE", "test_MAE", "test_R2",
            "orig_RMSE", "orig_MSE", "orig_MAE", "orig_RMSPE", "orig_R2"]]
          .rename(columns={"test_RMSE": "RMSE (log)", "test_MAE": "MAE (log)",
                           "test_R2": "R2 (log)", "orig_RMSE": "RMSE (asli)",
                           "orig_MSE": "MSE (asli)", "orig_MAE": "MAE (asli)",
                           "orig_RMSPE": "RMSPE (asli)", "orig_R2": "R2 (asli)"}))
    emit(t3.round(4), "tab3_rossmann_leakage_ablation",
         "Dampak kebocoran fitur Customers pada Rossmann Store Sales. "
         "V0 memakai Customers kontemporer yang tidak tersedia pada horizon peramalan.")
    display(t3)

## Tabel 4 - Ringkasan uji Diebold-Mariano

In [ ]:
dm_frames = []
for stem, label in [("exp01_pharma_daily_dm_test", "PharmaSales harian"),
                    ("exp02_pharma_weekly_dm_test", "PharmaSales mingguan"),
                    ("exp03_rossmann_leakage_ablation_dm_test", "Rossmann")]:
    path = RESULTS / f"{stem}.csv"
    if path.exists():
        d = pd.read_csv(path); d.insert(0, "eksperimen", label); dm_frames.append(d)

if dm_frames:
    t4 = pd.concat(dm_frames, ignore_index=True)
    emit(t4.set_index(["eksperimen"]), "tab4_diebold_mariano",
         "Uji Diebold-Mariano (koreksi Harvey-Leybourne-Newbold) metode usulan "
         "terhadap pembanding terkuat pada kondisi identik.")
    display(t4)
    if "p_value" in t4.columns:
        sig = t4[t4["p_value"] < 0.05]
        print(f"\nSelisih signifikan pada alpha=0.05: {len(sig)} dari {len(t4)} perbandingan")

## Tabel 5 - Kartu reproduktifitas

Tabel ini menjawab langsung butir "insufficient reproducibility": versi pustaka,
seed, rentang tanggal setiap blok, ukuran setiap blok, dan hyperparameter terpilih
per model - semuanya dibaca dari berkas hasil, bukan diketik ulang.

In [ ]:
frames = [d for d in [daily, weekly, rossman] if d is not None]
if frames:
    repro = pd.concat(frames, ignore_index=True)[
        ["category", "feature_set", "model", "n_features", "n_lags", "lag_rule",
         "n_train", "n_val", "n_test", "train_start", "train_end",
         "val_start", "val_end", "test_start", "test_end", "scaler", "seed",
         "n_grid", "params"]]
    emit(repro.set_index(["category", "feature_set", "model"]),
         "tab5_reproducibility_card",
         "Kartu reproduktifitas: konfigurasi lengkap setiap sel pada Tabel 1-3.")
    display(repro.head(20))

meta = {}
for stem in ["exp01_pharma_daily", "exp02_pharma_weekly",
             "exp03_rossmann_leakage_ablation"]:
    path = RESULTS / f"{stem}.meta.json"
    if path.exists():
        meta[stem] = json.loads(path.read_text())
(OUT / "environment.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(json.dumps(meta, indent=2)[:1500])

## Pemeriksaan konsistensi terakhir

Sebelum angka dipindahkan ke naskah, sel ini memverifikasi ulang aritmetikanya.
Reviewer menemukan RMSE 525,994 dilaporkan bersama MSE 276.669,25 (tidak mengkuadrat
secara eksak). Pemeriksaan ini membuat kesalahan sejenis mustahil lolos.

In [ ]:
problems = []
for name, df in [("daily", daily), ("weekly", weekly), ("rossmann", rossman)]:
    if df is None:
        continue
    for prefix in ["val_", "test_", "orig_"]:
        rmse_col, mse_col = f"{prefix}RMSE", f"{prefix}MSE"
        if rmse_col not in df.columns or mse_col not in df.columns:
            continue
        bad = df[~np.isclose(df[rmse_col] ** 2, df[mse_col], rtol=1e-9, atol=1e-12)]
        if len(bad):
            problems.append((name, prefix, len(bad)))
    # tidak boleh ada sel kosong pada kolom metrik utama
    empties = df[["test_RMSE", "test_MSE", "test_MAE"]].isna().sum().sum()
    if empties:
        problems.append((name, "sel metrik kosong", int(empties)))

if problems:
    print("MASALAH DITEMUKAN:")
    for p_ in problems:
        print("  ", p_)
else:
    print("Lolos: RMSE = sqrt(MSE) secara eksak di semua baris; "
          "tidak ada sel metrik yang kosong.")